# Generalisation study: does focused spatial attention help Mario agents on unseen levels?

Follow-up to `Mario_RL_Attention_Research.ipynb`. That study trained one agent per level and found that spatial self-attention neither sped up learning nor transferred to unseen levels, and that its attention was close to uniform. This notebook addresses the three causes it identified:

| Cause found in v1 | Change here |
|---|---|
| Single-level training: agents memorise one level | Train on 16 levels, test on 6 held-out levels |
| Policies tied to exact pixel positions | Random-shift augmentation with a DrAC consistency loss (Raileanu et al., 2021) |
| Attention spread uniformly over the screen | Sparse positional attention: 4 heads, learned 2D position embedding, each position attends to its top-8 matches |

Run headless with `NOTEBOOK=Mario_RL_Generalization.ipynb ./run_experiments.sh`. Outputs go to `results_generalization/` and `checkpoints_generalization/`.

## 1. Setup: install, versions, parameters & output folders

In [ ]:
import importlib.util, subprocess, sys

NEW_MARIO_STACK = sys.version_info >= (3, 13)
if any(importlib.util.find_spec(m) is None for m in ("stable_baselines3", "gym_super_mario_bros", "nes_py", "scipy")):
    pkgs = ["stable-baselines3", "pandas", "matplotlib", "pillow", "scipy"]
    pkgs += (["gym-super-mario-bros>=9.1"] if NEW_MARIO_STACK else
             ["gym-super-mario-bros<8", "nes-py<9", "gym==0.25.2", "numpy==1.26.4"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import json, os, platform, re, shutil, time, warnings
from collections import deque
from importlib.metadata import version as pkg_version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium
from gymnasium import spaces
import stable_baselines3 as sb3
from scipy import stats
from PIL import Image
from IPython.display import display, Image as IPyImage

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*out of date.*")
warnings.filterwarnings("ignore", message=".*step API.*")

def _v(pkg):
    try:
        return pkg_version(pkg)
    except PackageNotFoundError:
        return None

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
            "gymnasium": gymnasium.__version__, "stable_baselines3": sb3.__version__,
            "gym_super_mario_bros": _v("gym-super-mario-bros"), "nes_py": _v("nes-py"), "gym": _v("gym")}
LEGACY_API = int(VERSIONS["nes_py"].split(".")[0]) < 9
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items() if v))
print("CPU cores:", os.cpu_count(), "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if LEGACY_API and not np.__version__.startswith("1."):
    raise RuntimeError("nes-py < 9 needs NumPy 1.x: pip install numpy==1.26.4, then restart the kernel")

In [ ]:
# Parameters: papermill overrides these, e.g. -p TOTAL_TIMESTEPS 5000000 -y 'SEEDS: [42]'
TRAIN_LEVELS = ["1-1", "1-2", "2-3", "3-1", "3-3", "4-1", "4-3", "5-1", "5-2", "6-1", "6-2", "6-3", "7-1", "7-3", "8-2", "8-3"]
TEST_LEVELS = ["1-3", "2-1", "3-2", "4-2", "5-3", "8-1"]     # never seen in training
VARIANTS = ["cnn", "cnn_aug", "attn_aug", "sparse_aug"]       # optional: "impala_aug", "attn", "sparse"
SEEDS = [42, 123, 777]
TOTAL_TIMESTEPS = 3_000_000
N_ENVS = 8
TRAIN = True
TIME_BUDGET_HOURS = None
EVAL_EPISODES = 10                                # per seed on each held-out level
EVAL_TRAIN_EPISODES = 5                           # per seed on each training level (to measure the generalisation gap)
RANDOM_EPISODES = 10                              # random-policy reference, per seed and level
GIF_STEPS = 300
REEVALUATE = False                                # False: reuse saved evaluations and GIFs

In [ ]:
TITLE = "Generalisation of spatial attention in pixel-based PPO: Super Mario Bros. held-out levels"

# variant -> (encoder block, backbone, augmentation)
VARIANT_SPEC = {
    "cnn":        ("baseline", "nature", False),
    "cnn_aug":    ("baseline", "nature", True),
    "attn":       ("spatial",  "nature", False),
    "attn_aug":   ("spatial",  "nature", True),
    "sparse":     ("sparse",   "nature", False),
    "sparse_aug": ("sparse",   "nature", True),
    "impala_aug": ("baseline", "impala", True),
}
assert not set(TRAIN_LEVELS) & set(TEST_LEVELS), "a level cannot be both training and held-out"
assert all(v in VARIANT_SPEC for v in VARIANTS), f"unknown variant; choose from {list(VARIANT_SPEC)}"

CONFIG = {
    "train_levels": TRAIN_LEVELS, "test_levels": TEST_LEVELS, "variants": VARIANTS, "seeds": SEEDS,
    "total_timesteps": TOTAL_TIMESTEPS, "n_envs": N_ENVS, "train": TRAIN, "reevaluate": REEVALUATE,
    "rom_version": "v0", "frame_skip": 4, "frame_stack": 4, "reward_scale": 0.1,
    "checkpoint_every": 100_000, "keep_checkpoints": 3, "time_budget_hours": TIME_BUDGET_HOURS,
    "ppo": dict(learning_rate=1e-4, n_steps=512, batch_size=256, n_epochs=4, gamma=0.99, ent_coef=0.01, target_kl=None),
    "model": dict(features_dim=512, attn_d_model=64, se_reduction=16, sparse_heads=4, sparse_topk=8),
    "drac": dict(coef=0.1, pad=6),                # consistency-loss weight and max shift in pixels
    "eval_episodes": EVAL_EPISODES, "eval_train_episodes": EVAL_TRAIN_EPISODES, "random_episodes": RANDOM_EPISODES,
    "eval_deterministic": False, "smooth_window": 100, "gif_steps": GIF_STEPS, "gif_fps": 10,
}

ROOT = Path.cwd()
CKPT_DIR = ROOT / "checkpoints_generalization"
RESULTS = ROOT / "results_generalization"
DIRS = {k: RESULTS / k for k in ["figures", "tables", "gifs", "attention", "logs"]}
TRAIN_DEADLINE = time.time() + 3600 * TIME_BUDGET_HOURS if TIME_BUDGET_HOURS else None
for d in [CKPT_DIR, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

LABELS = {"cnn": "NatureCNN", "cnn_aug": "NatureCNN + aug", "attn": "Dense attention", "attn_aug": "Dense attention + aug",
          "sparse": "Sparse attention", "sparse_aug": "Sparse attention + aug (ours)", "impala_aug": "IMPALA-CNN + aug",
          "random": "Random policy"}
COLORS = {"cnn": "#8a8a85", "cnn_aug": "#2a78d6", "attn": "#c9a3e6", "attn_aug": "#eb6834", "sparse": "#9fd9c2",
          "sparse_aug": "#1baf7a", "impala_aug": "#6250d6", "random": "#bdbdb8"}

def run_name(variant, seed):
    return f"{variant}_s{seed}"

def experiment_matrix():
    return [(v, s) for s in SEEDS for v in VARIANTS]          # seed-major

ENV_INFO = {**VERSIONS, "cpu_count": os.cpu_count(),
            "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "platform": platform.platform()}
(RESULTS / "config.json").write_text(json.dumps({"title": TITLE, "config": CONFIG, "environment": ENV_INFO}, indent=2))
print(f"{len(experiment_matrix())} runs ({len(VARIANTS)} variants × {len(SEEDS)} seeds × {TOTAL_TIMESTEPS:,} steps), "
      f"{len(TRAIN_LEVELS)} training levels, {len(TEST_LEVELS)} held-out levels | results → {RESULTS}")

In [ ]:
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})

def save_fig(fig, name, formats=("png", "pdf")):
    for ext in formats:
        fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved → {RESULTS.name}/figures/{name}.{{{','.join(formats)}}}")

def _fmt(v, floatfmt):
    if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
        return "–"
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (float, np.floating)):
        return format(v, floatfmt)
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    return str(v)

def df_to_markdown(df, floatfmt=",.2f"):
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    rule = "|" + "|".join([":---"] + ["---:"] * (len(df.columns) - 1)) + "|"
    return "\n".join([head, rule, *["| " + " | ".join(_fmt(v, floatfmt) for v in r) + " |" for r in df.itertuples(index=False)]])

def df_to_latex(df, floatfmt=",.2f", caption="", label=""):
    def esc(s):
        s = str(s)
        for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"),
                     ("±", r"$\pm$"), ("≥", r"$\geq$"), ("×", r"$\times$"), ("−", r"$-$"), ("→", r"$\rightarrow$")]:
            s = s.replace(a, b)
        return s
    body = "\n".join(" & ".join(esc(_fmt(v, floatfmt)) for v in r) + r" \\" for r in df.itertuples(index=False))
    return ("\\begin{table}[t]\n\\centering\n" f"\\caption{{{esc(caption)}}}\n\\label{{tab:{label}}}\n"
            f"\\begin{{tabular}}{{l{'r' * (len(df.columns) - 1)}}}\n\\toprule\n"
            + " & ".join(esc(c) for c in df.columns) + " \\\\\n\\midrule\n" + body + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")

def save_table(df, name, caption=None, floatfmt=",.2f"):
    caption = caption or name
    df.to_csv(DIRS["tables"] / f"{name}.csv", index=False)
    (DIRS["tables"] / f"{name}.md").write_text(f"### {caption}\n\n{df_to_markdown(df, floatfmt)}\n")
    (DIRS["tables"] / f"{name}.tex").write_text(df_to_latex(df, floatfmt, caption, name))
    print(f"saved → {RESULTS.name}/tables/{name}.{{csv,md,tex}}")
    display(df)

def pm(values, fmt="{:,.2f}"):
    v = np.asarray(values, dtype=float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return "–"
    return fmt.format(v.mean()) if len(v) == 1 else f"{fmt.format(v.mean())} ± {fmt.format(v.std(ddof=1))}"

def bootstrap_diff_ci(a, b, n=10_000, seed=0):
    rng = np.random.default_rng(seed)
    d = rng.choice(a, (n, len(a))).mean(1) - rng.choice(b, (n, len(b))).mean(1)
    return np.percentile(d, [2.5, 97.5])

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    order = np.argsort(p[ok]); m = ok.sum()
    out[np.flatnonzero(ok)[order]] = np.maximum.accumulate((m - np.arange(m)) * p[ok][order]).clip(max=1)
    return out

## 2. Environments: single level (evaluation) and multi-level (training)

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

ACTION_NAMES = ["+".join(a) for a in SIMPLE_MOVEMENT]

def make_mario(level, show=False):
    kwargs = {} if LEGACY_API or not show else {"render_mode": "human"}
    return JoypadSpace(gym_super_mario_bros.make(f"SuperMarioBros-{level}-{CONFIG['rom_version']}", **kwargs), SIMPLE_MOVEMENT)

def mario_reset(env):
    out = env.reset()
    return out[0] if isinstance(out, tuple) else out

def mario_step(env, action):
    out = env.step(action)
    if len(out) == 4:
        frame, reward, done, info = out
        return frame, reward, done, False, info
    return out

def prep(frame):
    small = frame[::2, ::2].astype(np.float32)                          # 240×256 → 120×128
    return (small[..., 0] * 0.299 + small[..., 1] * 0.587 + small[..., 2] * 0.114).astype(np.uint8)[:, :, None]


class MarioEnv(gymnasium.Env):
    """One level, or a set of levels with a uniformly random level at every reset (one life per episode).
    Frame skip 4 and 120×128 grayscale, as in v1. `info["level"]` names the level being played."""
    metadata = {"render_modes": []}
    render_mode = None

    def __init__(self, levels, skip=4, show=False):
        super().__init__()
        self.levels = [levels] if isinstance(levels, str) else list(levels)
        self.skip, self.show, self.emus, self.level = skip, show, {}, None
        self.action_space = spaces.Discrete(len(SIMPLE_MOVEMENT))
        self.observation_space = spaces.Box(0, 255, (120, 128, 1), np.uint8)

    def _emu(self, level):
        if level not in self.emus:
            self.emus[level] = make_mario(level, self.show)
        return self.emus[level]

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.level = self.levels[self.np_random.integers(len(self.levels))]
        return prep(mario_reset(self._emu(self.level))), {"level": self.level}

    def step(self, action):
        env, total, terminated, truncated, info = self.emus[self.level], 0.0, False, False, {}
        for _ in range(self.skip):
            frame, reward, terminated, truncated, info = mario_step(env, int(action))
            total += reward
            if self.show and LEGACY_API:
                env.render()
            if terminated or truncated:
                break
        info = dict(info, level=self.level)
        return prep(frame), float(total), bool(terminated), bool(truncated), info

    def close(self):
        for e in self.emus.values():
            e.close()

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecFrameStack

def make_vec_env(levels, n_envs=CONFIG["n_envs"], show=False, monitor=True, reward_scale=1.0):
    def make():
        env = MarioEnv(levels, skip=CONFIG["frame_skip"], show=show)
        if monitor:
            env = Monitor(env, info_keywords=("level",))
        if reward_scale != 1.0:
            env = gymnasium.wrappers.TransformReward(env, lambda r: r * reward_scale)
        return env
    venv = DummyVecEnv([make]) if n_envs == 1 else SubprocVecEnv([make] * n_envs, start_method="fork")
    return VecFrameStack(venv, CONFIG["frame_stack"], channels_order="last")

fig, axes = plt.subplots(2, 11, figsize=(22, 4.4))
for ax, lvl in zip(axes.flat, TRAIN_LEVELS + TEST_LEVELS):
    e = make_mario(lvl); mario_reset(e)
    for _ in range(40):
        f, *_ = mario_step(e, 1)
    e.close()
    ax.imshow(f); ax.axis("off")
    ax.set_title(f"{lvl} ({'test' if lvl in TEST_LEVELS else 'train'})", fontsize=9,
                 color="#c03a2b" if lvl in TEST_LEVELS else "black")
for ax in axes.flat[len(TRAIN_LEVELS + TEST_LEVELS):]:
    ax.axis("off")
fig.suptitle("Training levels and held-out test levels (red)")
fig.tight_layout(); save_fig(fig, "levels"); plt.show()

## 3. Encoders: NatureCNN / IMPALA-CNN + optional block (SE, dense attention, sparse positional attention)

In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticCnnPolicy

class SpatialSelfAttention(nn.Module):
    """Dense single-head self-attention over all positions, γ-gated (as in v1)."""
    def __init__(self, channels, hw=None, d_model=64, **_):
        super().__init__()
        self.d = d_model
        self.q = nn.Conv2d(channels, d_model, 1, bias=False); self.k = nn.Conv2d(channels, d_model, 1, bias=False)
        self.v = nn.Conv2d(channels, d_model, 1, bias=False); self.out = nn.Conv2d(d_model, channels, 1, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.store_attn, self.last_attn = False, None

    def forward(self, x):
        B, C, H, W = x.shape; N = H * W
        q, k, v = self.q(x).view(B, self.d, N), self.k(x).view(B, self.d, N), self.v(x).view(B, self.d, N)
        A = torch.softmax(torch.bmm(q.transpose(1, 2), k) * self.d ** -0.5, dim=-1)
        if self.store_attn:
            self.last_attn = A.detach().cpu()
        return self.gamma * self.out(torch.bmm(v, A.transpose(1, 2)).view(B, self.d, H, W)) + x


class SparsePositionalAttention(nn.Module):
    """Proposed block. Multi-head attention where (i) queries and keys see a learned 2D position embedding,
    so attention can depend on WHERE things are, and (ii) each query keeps only its top-k keys, so attention
    cannot spread uniformly (the failure observed in v1). Values carry content only. γ-gated residual."""
    def __init__(self, channels, hw, d_model=64, heads=4, topk=8, **_):
        super().__init__()
        self.h, self.dh, self.topk = heads, d_model // heads, topk
        self.pos = nn.Parameter(torch.zeros(1, channels, *hw))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.q = nn.Conv2d(channels, d_model, 1, bias=False); self.k = nn.Conv2d(channels, d_model, 1, bias=False)
        self.v = nn.Conv2d(channels, d_model, 1, bias=False); self.out = nn.Conv2d(d_model, channels, 1, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.store_attn, self.last_attn = False, None

    def forward(self, x):
        B, C, H, W = x.shape; N = H * W
        xp = x + self.pos
        q = self.q(xp).view(B, self.h, self.dh, N); k = self.k(xp).view(B, self.h, self.dh, N)
        v = self.v(x).view(B, self.h, self.dh, N)
        logits = torch.einsum("bhdn,bhdm->bhnm", q, k) * self.dh ** -0.5
        if self.topk < N:
            kth = logits.topk(self.topk, dim=-1).values[..., -1:]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        A = logits.softmax(-1)
        if self.store_attn:
            self.last_attn = A.mean(1).detach().cpu()              # head-averaged, for visualisation
        out = torch.einsum("bhnm,bhdm->bhdn", A, v).reshape(B, self.h * self.dh, H, W)
        return self.gamma * self.out(out) + x


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16, **_):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.ReLU(),
                                nn.Linear(channels // reduction, channels), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x.mean(dim=(2, 3)))[:, :, None, None]


class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(c, c, 3, padding=1), nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        return x + self.c2(F.relu(self.c1(F.relu(x))))


def impala_cnn(in_ch, widths=(16, 32, 32)):
    """IMPALA ResNet encoder (Espeholt et al., 2018), the standard strong encoder for Procgen generalisation."""
    layers, c = [], in_ch
    for w in widths:
        layers += [nn.Conv2d(c, w, 3, padding=1), nn.MaxPool2d(3, stride=2, padding=1), ResidualBlock(w), ResidualBlock(w)]
        c = w
    return nn.Sequential(*layers, nn.ReLU())


class Encoder(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512, block="baseline", backbone="nature", attn_d_model=64,
                 se_reduction=16, sparse_heads=4, sparse_topk=8):
        super().__init__(observation_space, features_dim)
        c_in = observation_space.shape[0]
        self.cnn = (nn.Sequential(nn.Conv2d(c_in, 32, 8, stride=4), nn.ReLU(), nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
                                  nn.Conv2d(64, 64, 3, stride=1), nn.ReLU())
                    if backbone == "nature" else impala_cnn(c_in))
        with torch.no_grad():
            _, c, h, w = self.cnn(torch.zeros(1, *observation_space.shape)).shape
        self.feat_chw = (c, h, w)
        self.block = {"baseline": lambda: None,
                      "channel": lambda: SEBlock(c, se_reduction),
                      "spatial": lambda: SpatialSelfAttention(c, (h, w), attn_d_model),
                      "sparse": lambda: SparsePositionalAttention(c, (h, w), attn_d_model, sparse_heads, sparse_topk)}[block]()
        self.linear = nn.Sequential(nn.Flatten(), nn.Linear(c * h * w, features_dim), nn.ReLU())

    def forward(self, obs):
        x = self.cnn(obs)
        if self.block is not None:
            x = self.block(x)
        return self.linear(x)

OBS_SPACE = spaces.Box(0, 255, (CONFIG["frame_stack"], 120, 128), np.uint8)
ACT_SPACE = spaces.Discrete(len(SIMPLE_MOVEMENT))

def policy_kwargs(variant):
    block, backbone, _ = VARIANT_SPEC[variant]
    m = CONFIG["model"]
    return dict(net_arch=[], features_extractor_class=Encoder,
                features_extractor_kwargs=dict(block=block, backbone=backbone, features_dim=m["features_dim"],
                                               attn_d_model=m["attn_d_model"], se_reduction=m["se_reduction"],
                                               sparse_heads=m["sparse_heads"], sparse_topk=m["sparse_topk"]))

def build_policy(variant):
    return ActorCriticCnnPolicy(OBS_SPACE, ACT_SPACE, lr_schedule=lambda _: 1e-4, **policy_kwargs(variant))

n_params = lambda m: sum(p.numel() for p in m.parameters())
rows = []
for v in dict.fromkeys(VARIANTS + ["cnn", "attn_aug", "sparse_aug"]):
    pol = build_policy(v); blk = pol.features_extractor.block
    rows.append({"Variant": LABELS[v], "Encoder": VARIANT_SPEC[v][1], "Block": VARIANT_SPEC[v][0],
                 "Augmentation": VARIANT_SPEC[v][2], "Block params": n_params(blk) if blk is not None else 0,
                 "Total params": n_params(pol)})
save_table(pd.DataFrame(rows), "architecture_params", "Variants and model size")

## 4. Augmentation-consistency PPO (DrAC)

Random shift: every observation in a minibatch is shifted by up to ±6 pixels (edges replicated). The PPO loss uses the original observations; an extra loss asks the policy and value to be the same for the shifted copy (Raileanu et al., 2021):

`L = L_PPO + α · ( KL[π(·|s) ‖ π(·|shift(s))] + (V(shift(s)) − V(s))² )`, α = 0.1, with π(·|s) and V(s) held fixed.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.utils import explained_variance

def random_shift(x, pad):
    """Shift each image by an independent random integer offset in [-pad, pad] (edges replicated)."""
    dtype = x.dtype
    x = x.float()
    n, c, h, w = x.shape
    x = F.pad(x, (pad, pad, pad, pad), mode="replicate")
    H, W = h + 2 * pad, w + 2 * pad
    ys = torch.linspace(-1 + 1 / H, 1 - 1 / H, H, device=x.device)[:h]
    xs = torch.linspace(-1 + 1 / W, 1 - 1 / W, W, device=x.device)[:w]
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).expand(n, h, w, 2)
    shift = torch.randint(0, 2 * pad + 1, (n, 1, 1, 2), device=x.device, dtype=x.dtype)
    shift = shift * torch.tensor([2.0 / W, 2.0 / H], device=x.device, dtype=x.dtype)
    return F.grid_sample(x, grid + shift, mode="nearest", padding_mode="border", align_corners=False).to(dtype)


class DrACPPO(PPO):
    """PPO (Stable-Baselines3 2.x train loop) plus the DrAC augmentation-consistency loss. aug_coef = 0 is plain PPO."""
    aug_coef: float = 0.0
    aug_pad: int = 6

    def _forward(self, obs):
        """One encoder pass -> (action distribution, values); same computation as evaluate_actions."""
        pol = self.policy
        latent_pi, latent_vf = pol.mlp_extractor(pol.extract_features(obs))
        return pol._get_action_dist_from_latent(latent_pi), pol.value_net(latent_vf)

    def train(self) -> None:
        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)
        clip_range = self.clip_range(self._current_progress_remaining)
        entropy_losses, pg_losses, value_losses, clip_fractions, aug_losses = [], [], [], [], []
        continue_training = True
        for epoch in range(self.n_epochs):
            approx_kl_divs = []
            for rollout_data in self.rollout_buffer.get(self.batch_size):
                actions = rollout_data.actions.long().flatten()
                dist, values = self._forward(rollout_data.observations)
                log_prob, entropy = dist.log_prob(actions), dist.entropy()
                values = values.flatten()
                advantages = rollout_data.advantages
                if self.normalize_advantage and len(advantages) > 1:
                    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
                ratio = torch.exp(log_prob - rollout_data.old_log_prob)
                policy_loss = -torch.min(advantages * ratio,
                                         advantages * torch.clamp(ratio, 1 - clip_range, 1 + clip_range)).mean()
                pg_losses.append(policy_loss.item())
                clip_fractions.append(torch.mean((torch.abs(ratio - 1) > clip_range).float()).item())
                value_loss = F.mse_loss(rollout_data.returns, values)
                value_losses.append(value_loss.item())
                entropy_loss = -torch.mean(entropy)
                entropy_losses.append(entropy_loss.item())
                loss = policy_loss + self.ent_coef * entropy_loss + self.vf_coef * value_loss

                if self.aug_coef > 0:
                    # targets = this minibatch's own outputs, held fixed (no second pass on the original frames)
                    target_dist = torch.distributions.Categorical(logits=dist.distribution.logits.detach())
                    target_v = values.detach()
                    aug_dist, aug_v = self._forward(random_shift(rollout_data.observations, self.aug_pad))
                    aug_loss = (torch.distributions.kl_divergence(target_dist, aug_dist.distribution).mean()
                                + F.mse_loss(aug_v.flatten(), target_v))
                    loss = loss + self.aug_coef * aug_loss
                    aug_losses.append(aug_loss.item())

                with torch.no_grad():
                    log_ratio = log_prob - rollout_data.old_log_prob
                    approx_kl_div = torch.mean((torch.exp(log_ratio) - 1) - log_ratio).cpu().numpy()
                    approx_kl_divs.append(approx_kl_div)
                if self.target_kl is not None and approx_kl_div > 1.5 * self.target_kl:
                    continue_training = False
                    break
                self.policy.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.policy.optimizer.step()
            self._n_updates += 1
            if not continue_training:
                break

        self.logger.record("train/entropy_loss", np.mean(entropy_losses))
        self.logger.record("train/policy_gradient_loss", np.mean(pg_losses))
        self.logger.record("train/value_loss", np.mean(value_losses))
        self.logger.record("train/approx_kl", np.mean(approx_kl_divs))
        self.logger.record("train/clip_fraction", np.mean(clip_fractions))
        self.logger.record("train/explained_variance",
                           explained_variance(self.rollout_buffer.values.flatten(), self.rollout_buffer.returns.flatten()))
        if aug_losses:
            self.logger.record("train/aug_consistency_loss", np.mean(aug_losses))
        self.logger.record("train/n_updates", self._n_updates, exclude="tensorboard")

_x = torch.randint(0, 256, (4, 4, 120, 128), dtype=torch.uint8)       # the rollout buffer stores uint8 frames
_y = random_shift(_x, CONFIG["drac"]["pad"])
_xp = F.pad(_x, (6,) * 4, mode="replicate")
assert all(any(torch.equal(_y[i], _xp[i, :, a:a + 120, b:b + 128]) for a in range(13) for b in range(13)) for i in range(4))
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
e = MarioEnv(TRAIN_LEVELS[0]); o, _ = e.reset(seed=0); e.close()
img = torch.tensor(o[None, :, :, 0][None]).float()
for ax, k in zip(axes, range(4)):
    ax.imshow((img if k == 0 else random_shift(img, CONFIG["drac"]["pad"]))[0, 0], cmap="gray"); ax.axis("off")
    ax.set_title("original" if k == 0 else f"random shift {k}")
fig.tight_layout(); save_fig(fig, "augmentation_examples"); plt.show()

## 5. Logging, training & resume

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.logger import configure
from stable_baselines3.common.utils import set_random_seed

class EpisodeCSVLogger(BaseCallback):
    FIELDS = ["episode", "timesteps", "level", "reward", "length", "max_x_pos", "flag_get", "unix_time"]

    def __init__(self, csv_path, name="", print_every=10):
        super().__init__()
        self.csv_path, self.name, self.print_every, self.f = Path(csv_path), name, print_every, None

    def _on_training_start(self):
        new = not self.csv_path.exists() or self.csv_path.stat().st_size == 0
        if new:
            self.n_episodes = 0
        else:
            df = pd.read_csv(self.csv_path, dtype={"level": str})
            if (df.timesteps > self.model.num_timesteps).any():
                df = df[df.timesteps <= self.model.num_timesteps]; df.to_csv(self.csv_path, index=False)
            self.n_episodes = len(df)
        self.f = open(self.csv_path, "a", buffering=1)
        if new:
            self.f.write(",".join(self.FIELDS) + "\n")
        self.max_x = np.zeros(self.training_env.num_envs)
        self.recent = deque(maxlen=200)
        self.n_rollouts, self.t0, self.ts0 = 0, time.time(), self.model.num_timesteps

    def _on_step(self):
        for i, info in enumerate(self.locals["infos"]):
            if info.get("x_pos") is not None:
                self.max_x[i] = max(self.max_x[i], info["x_pos"])
            if "episode" in info:
                self.n_episodes += 1
                lvl, flag = info.get("level", ""), int(bool(info.get("flag_get", False)))
                self.f.write(",".join(map(str, [self.n_episodes, self.num_timesteps, lvl, f"{info['episode']['r']:.2f}",
                                                info["episode"]["l"], int(self.max_x[i]), flag, f"{time.time():.1f}"])) + "\n")
                self.recent.append((self.max_x[i] / RANDOM_MEAN.get(lvl, np.nan), flag))
                self.max_x[i] = 0
        return True

    def _on_rollout_end(self):
        self.n_rollouts += 1
        if self.print_every and self.n_rollouts % self.print_every == 0 and self.recent:
            r, fl = np.array(self.recent).T
            fps = (self.model.num_timesteps - self.ts0) / max(time.time() - self.t0, 1e-9)
            print(f"[{self.name}] {self.model.num_timesteps:>10,} steps | {fps:5.0f} fps | last {len(r)} eps: "
                  f"progress {np.nanmean(r):.2f}× random | flags {int(fl.sum())}", flush=True)

    def close(self):
        if self.f is not None and not self.f.closed:
            self.f.close()

    def _on_training_end(self):
        self.close()


class GammaCallback(BaseCallback):
    def _on_step(self):
        return True

    def _on_rollout_end(self):
        g = getattr(getattr(self.model.policy.features_extractor, "block", None), "gamma", None)
        if g is not None:
            self.logger.record("block/gamma", g.item())


class DeadlineCallback(BaseCallback):
    def __init__(self, deadline):
        super().__init__(); self.deadline = deadline

    def _on_step(self):
        return time.time() < self.deadline


def _run_info_path(run):
    return DIRS["logs"] / run / "run_info.json"

def _run_info(run):
    p = _run_info_path(run)
    return json.loads(p.read_text()) if p.exists() else {}

def _step_ckpts(run):
    return sorted(((int(m.group(1)), p) for p in (CKPT_DIR / run).glob("*.zip")
                   if (m := re.search(r"_(\d+)_steps\.zip$", p.name))), key=lambda c: c[0])

def resolve_checkpoint(run):
    cands = _step_ckpts(run)
    final = CKPT_DIR / run / f"{run}_final.zip"
    if final.exists():
        cands.append((_run_info(run).get("timesteps", 0), final))
    return max(cands, key=lambda c: c[0])[1] if cands else None

def load_run(run, env=None):
    path = resolve_checkpoint(run)
    if path is None:
        raise FileNotFoundError(f"no checkpoint for '{run}'")
    return DrACPPO.load(path, env=env, device="auto")

def train_run(variant, seed, total_timesteps=CONFIG["total_timesteps"]):
    run = run_name(variant, seed)
    ckpt_dir, log_dir = CKPT_DIR / run, DIRS["logs"] / run
    ckpt_dir.mkdir(parents=True, exist_ok=True); log_dir.mkdir(parents=True, exist_ok=True)
    info = _run_info(run) or {"run": run, "variant": variant, "seed": seed, "sessions": 0, "timesteps": 0,
                              "logged_steps": 0, "train_seconds": 0.0}
    if info.get("timesteps", 0) >= total_timesteps:
        print(f"[{run}] already trained to {info['timesteps']:,} steps, skipping"); return None
    extra = []
    if TRAIN_DEADLINE is not None:
        order = [run_name(*r) for r in experiment_matrix()]
        left = [r for r in order[order.index(run):] if r == run or _run_info(r).get("timesteps", 0) < total_timesteps]
        run_deadline = time.time() + (TRAIN_DEADLINE - time.time()) / len(left)
        if run_deadline - time.time() < 60:
            print(f"[{run}] time budget used up, skipping"); return None
        extra.append(DeadlineCallback(run_deadline))

    env = make_vec_env(TRAIN_LEVELS, CONFIG["n_envs"], reward_scale=CONFIG["reward_scale"])
    start = resolve_checkpoint(run)
    if start is not None:
        model = DrACPPO.load(start, env=env, device="auto")
        print(f"[{run}] loaded {start.name} ({model.num_timesteps:,} steps)")
    else:
        set_random_seed(seed)
        model = DrACPPO("CnnPolicy", env, policy_kwargs=policy_kwargs(variant), seed=seed, verbose=0, device="auto",
                        **CONFIG["ppo"])
        print(f"[{run}] new model on {model.device}")
    model.aug_coef = CONFIG["drac"]["coef"] if VARIANT_SPEC[variant][2] else 0.0
    model.aug_pad = CONFIG["drac"]["pad"]

    info["sessions"] += 1
    model.set_logger(configure(str(log_dir / f"sb3_session{info['sessions']}"), ["csv"]))
    ep_logger = EpisodeCSVLogger(log_dir / "episodes.csv", name=run)
    callbacks = [CheckpointCallback(save_freq=max(CONFIG["checkpoint_every"] // CONFIG["n_envs"], 1),
                                    save_path=str(ckpt_dir), name_prefix=run), ep_logger, GammaCallback(), *extra]
    t0, steps0 = time.time(), model.num_timesteps
    try:
        model.learn(total_timesteps=total_timesteps - model.num_timesteps, callback=callbacks, reset_num_timesteps=False)
        if model.num_timesteps >= total_timesteps:
            model.save(ckpt_dir / f"{run}_final.zip"); print(f"[{run}] done: {model.num_timesteps:,} steps")
        else:
            model.save(ckpt_dir / f"{run}_{model.num_timesteps}_steps.zip")
            print(f"[{run}] time budget reached at {model.num_timesteps:,} steps; re-run to resume")
    except KeyboardInterrupt:
        model.save(ckpt_dir / f"{run}_{model.num_timesteps}_steps.zip")
        print(f"\n[{run}] interrupted at {model.num_timesteps:,} steps, checkpoint saved"); raise
    finally:
        ep_logger.close()
        g = getattr(getattr(model.policy.features_extractor, "block", None), "gamma", None)
        info.update(timesteps=int(model.num_timesteps), logged_steps=info["logged_steps"] + int(model.num_timesteps - steps0),
                    train_seconds=round(info["train_seconds"] + time.time() - t0, 1), policy_params=n_params(model.policy),
                    gamma=None if g is None else round(g.item(), 5))
        info["train_fps"] = round(info["logged_steps"] / max(info["train_seconds"], 1e-9), 1)
        _run_info_path(run).write_text(json.dumps(info, indent=2))
        for _, p in _step_ckpts(run)[:-CONFIG["keep_checkpoints"]]:
            p.unlink()
        env.close()
    return model

## 6. Random-policy reference on every level

In [ ]:
class RandomAgent:
    def predict(self, obs, deterministic=False):
        return np.random.randint(ACT_SPACE.n, size=(obs.shape[0],)), None

def evaluate(agent, level, n_episodes, deterministic=False, max_steps=3000, seed=0):
    set_random_seed(seed)
    env = make_vec_env(level, 1, monitor=False)
    rows, obs = [], env.reset()
    for ep in range(1, n_episodes + 1):
        total, max_x = 0.0, 0
        for t in range(1, max_steps + 1):
            action, _ = agent.predict(obs, deterministic=deterministic)
            obs, reward, dones, infos = env.step(action)
            total += float(reward[0]); max_x = max(max_x, infos[0].get("x_pos", 0))
            if dones[0]:
                break
        else:
            obs = env.reset()
        rows.append({"episode": ep, "reward": total, "steps": t, "max_x_pos": max_x,
                     "flag_get": int(bool(infos[0].get("flag_get", False)))})
    env.close()
    return pd.DataFrame(rows)

RANDOM_FILE = DIRS["tables"] / "random_reference_episodes.csv"
ALL_LEVELS = TRAIN_LEVELS + TEST_LEVELS
rnd = pd.read_csv(RANDOM_FILE, dtype={"level": str}) if RANDOM_FILE.exists() and not REEVALUATE else pd.DataFrame()
missing = [(l, s) for l in ALL_LEVELS for s in SEEDS
           if not len(rnd) or not ((rnd.level == l) & (rnd.seed == s)).any()]
if missing:
    new = [evaluate(RandomAgent(), l, RANDOM_EPISODES, seed=s).assign(level=l, seed=s) for l, s in missing]
    rnd = pd.concat([rnd, *new], ignore_index=True)
    rnd.to_csv(RANDOM_FILE, index=False)
    print(f"saved → {RESULTS.name}/tables/random_reference_episodes.csv ({len(missing)} level-seed pairs added)")
else:
    print("using saved random-policy reference")
RANDOM_MEAN = rnd.groupby("level").max_x_pos.mean().to_dict()
save_table(pd.DataFrame({"Level": ALL_LEVELS, "Split": ["train" if l in TRAIN_LEVELS else "test" for l in ALL_LEVELS],
                         "Random mean max x": [RANDOM_MEAN[l] for l in ALL_LEVELS]}),
           "random_reference", "Random-policy mean max x per level (normaliser for all scores)", ",.0f")

## 7. Train

In [ ]:
if TRAIN:
    for variant, seed in experiment_matrix():
        train_run(variant, seed)
else:
    print("TRAIN = False: using existing checkpoints and logs")

## 8. Evaluate on training levels and held-out levels

Score = evaluation max x divided by the random policy's mean max x on the same level (1.0 = random play), averaged over levels. Generalisation gap = training-level score − held-out score.

In [ ]:
EVAL_FILE = DIRS["tables"] / "eval_episodes.csv"
ev = pd.read_csv(EVAL_FILE, dtype={"level": str}) if EVAL_FILE.exists() and not REEVALUATE else pd.DataFrame()
have = set(map(tuple, ev[["variant", "seed", "level"]].values)) if len(ev) else set()
for variant, seed in experiment_matrix():
    run = run_name(variant, seed)
    if resolve_checkpoint(run) is None:
        continue
    todo = [l for l in ALL_LEVELS if (variant, seed, l) not in have]
    if not todo:
        continue
    model = load_run(run)
    for level in todo:
        n = EVAL_EPISODES if level in TEST_LEVELS else EVAL_TRAIN_EPISODES
        df = evaluate(model, level, n, CONFIG["eval_deterministic"], seed=seed)
        ev = pd.concat([ev, df.assign(variant=variant, seed=seed, level=level,
                                      split="test" if level in TEST_LEVELS else "train")], ignore_index=True)
        ev.to_csv(EVAL_FILE, index=False)
    print(f"[eval] {run}: done", flush=True)

ev["ratio"] = ev.max_x_pos / ev.level.map(RANDOM_MEAN)
LEVEL_SCORE = ev.groupby(["variant", "seed", "split", "level"]).ratio.mean().reset_index()
SCORES = LEVEL_SCORE.groupby(["variant", "seed", "split"]).ratio.mean().unstack("split").reset_index()
SCORES["gap"] = SCORES["train"] - SCORES["test"]
SCORES.to_csv(DIRS["tables"] / "scores_per_seed.csv", index=False)

rows = []
for v in [v for v in VARIANTS if v in set(SCORES.variant)]:
    d = SCORES[SCORES.variant == v]; e = ev[ev.variant == v]
    rows.append({"Variant": LABELS[v], "Seeds": len(d),
                 "Train levels (× random)": pm(d["train"]), "Held-out levels (× random)": pm(d["test"]),
                 "Generalisation gap": pm(d["gap"]), "Held-out flags": int(e[e.split == "test"].flag_get.sum())})
save_table(pd.DataFrame(rows), "generalization_summary",
           f"Mean max x relative to random play, mean ± std over seeds ({len(TRAIN_LEVELS)} training, "
           f"{len(TEST_LEVELS)} held-out levels)")

In [ ]:
vs = [v for v in VARIANTS if v in set(SCORES.variant)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.6]})
ax = axes[0]
for i, v in enumerate(vs):
    d = SCORES[SCORES.variant == v]
    for j, (split, mk) in enumerate([("train", "o"), ("test", "s")]):
        x = i + (j - 0.5) * 0.32
        ax.scatter(np.full(len(d), x), d[split], color=COLORS[v], marker=mk, s=28, zorder=3, edgecolor="white", linewidth=0.6)
        ax.hlines(d[split].mean(), x - 0.13, x + 0.13, color="#2b2b28", lw=1.4, zorder=4)
ax.axhline(1, color="#8a8a85", ls=":", lw=1)
ax.set_xticks(range(len(vs)), [LABELS[v] for v in vs], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("max x / random policy"); ax.set_title("Training levels (○) vs held-out levels (□)")
ax = axes[1]
M = np.array([[LEVEL_SCORE[(LEVEL_SCORE.variant == v) & (LEVEL_SCORE.level == l)].ratio.mean() for l in TEST_LEVELS] for v in vs])
from matplotlib.colors import TwoSlopeNorm
im = ax.imshow(M, cmap="RdBu", aspect="auto",
               norm=TwoSlopeNorm(vmin=min(0.3, np.nanmin(M)), vcenter=1.0, vmax=max(3.0, np.nanmax(M))))
for i in range(len(vs)):
    for j in range(len(TEST_LEVELS)):
        ax.text(j, i, f"{M[i, j]:.1f}×", ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(TEST_LEVELS)), TEST_LEVELS); ax.set_yticks(range(len(vs)), [LABELS[v] for v in vs], fontsize=8)
ax.set_title("Held-out levels, per level (1.0× = random play)")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout(); save_fig(fig, "generalization"); plt.show()

PAIRS = [(a, b) for a, b in [("sparse_aug", "attn_aug"), ("sparse_aug", "cnn_aug"), ("attn_aug", "cnn_aug"),
                             ("cnn_aug", "cnn"), ("impala_aug", "cnn_aug"), ("sparse_aug", "impala_aug")]
         if a in vs and b in vs]
st = []
for metric in ["test", "train", "gap"]:
    for a, b in PAIRS:
        va = SCORES[SCORES.variant == a].sort_values("seed")[metric].to_numpy()
        vb = SCORES[SCORES.variant == b].sort_values("seed")[metric].to_numpy()
        if len(va) < 2 or len(vb) < 2:
            continue
        lo, hi = bootstrap_diff_ci(va, vb)
        st.append({"Metric": metric, "Comparison": f"{a} − {b}", "Mean A": va.mean(), "Mean B": vb.mean(),
                   "Difference": va.mean() - vb.mean(), "95% CI low": lo, "95% CI high": hi,
                   "Welch p": stats.ttest_ind(va, vb, equal_var=False).pvalue})
if st:
    st = pd.DataFrame(st)
    st["Holm p"] = np.nan
    for m in st.Metric.unique():
        st.loc[st.Metric == m, "Holm p"] = holm(st.loc[st.Metric == m, "Welch p"])
    save_table(st, "generalization_statistics", "Pairwise differences in score (× random), bootstrap 95% CI over seeds, "
               "Welch t-test, Holm-corrected within each metric", ",.3g")
else:
    print("statistics need ≥ 2 seeds per variant")

## 9. Learning curves (normalised by the random policy on each level)

In [ ]:
def load_episodes(v):
    dfs = []
    for s in SEEDS:
        p = DIRS["logs"] / run_name(v, s) / "episodes.csv"
        if p.exists() and p.stat().st_size > 0:
            df = pd.read_csv(p, dtype={"level": str})
            if len(df):
                dfs.append(df.assign(seed=s))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

W = CONFIG["smooth_window"]
fig, ax = plt.subplots(figsize=(8, 4))
for v in VARIANTS:
    ep = load_episodes(v)
    if not len(ep):
        continue
    ep["ratio"] = ep.max_x_pos / ep.level.map(RANDOM_MEAN)
    grid = np.linspace(0, ep.timesteps.max(), 300); ys = []
    for _, g in ep.groupby("seed"):
        g = g.sort_values("timesteps").assign(r=lambda d: d.ratio.rolling(W, min_periods=1).mean()).drop_duplicates("timesteps", keep="last")
        ys.append(np.interp(grid, g.timesteps, g.r, left=np.nan, right=np.nan))
    ys = np.array(ys); mu = np.nanmean(ys, 0)
    ax.plot(grid / 1e6, mu, color=COLORS[v], lw=2, label=f"{LABELS[v]} (n={len(ys)})")
    if len(ys) > 1:
        sd = np.nanstd(ys, 0); ax.fill_between(grid / 1e6, mu - sd, mu + sd, color=COLORS[v], alpha=0.15)
ax.axhline(1, color="#8a8a85", ls=":", lw=1)
ax.set_xlabel("environment steps (M)"); ax.set_ylabel(f"max x / random ({W}-episode rolling mean)")
ax.set_title("Training progress on the 16 training levels"); ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "learning_curves"); plt.show()

prog = []
for v in VARIANTS:
    for s in SEEDS:
        fs = sorted((DIRS["logs"] / run_name(v, s)).glob("sb3_session*/progress.csv"))
        dfs = [pd.read_csv(f) for f in fs if f.stat().st_size > 0]
        if dfs:
            prog.append(pd.concat(dfs).assign(variant=v, seed=s))
if prog:
    prog = pd.concat(prog)
    cols = [("train/entropy_loss", "policy entropy", -1), ("train/approx_kl", "approx. KL", 1),
            ("train/explained_variance", "explained variance", 1), ("block/gamma", "attention gate γ", 1)]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.4))
    for ax, (col, title, sign) in zip(axes, cols):
        if col not in prog:
            ax.set_visible(False); continue
        for (v, s), d in prog.groupby(["variant", "seed"]):
            d = d[["time/total_timesteps", col]].dropna()
            if len(d):
                ax.plot(d.iloc[:, 0] / 1e6, sign * d[col], color=COLORS[v], lw=1, alpha=0.8, label=LABELS[v] if s == SEEDS[0] else None)
        ax.set_title(title); ax.set_xlabel("environment steps (M)")
    axes[0].legend(fontsize=7)
    fig.tight_layout(); save_fig(fig, "training_diagnostics"); plt.show()

## 10. What the attention blocks look at (held-out level)

In [ ]:
def attention_saliency(A, hw, out_hw):
    sal = A.mean(dim=1).view(1, 1, *hw).float()
    sal = F.interpolate(sal, size=out_hw, mode="bilinear", align_corners=False).squeeze().numpy()
    lo, hi = sal.min(), sal.max()
    return ((sal - lo) / (hi - lo) if hi > lo else np.zeros_like(sal)).astype(np.float32)

def entropies(A):
    a = A[0].double().clamp_min(1e-12); log_n = np.log(a.shape[-1]); p = a.mean(0)
    return float((-(a * a.log()).sum(-1)).mean() / log_n), float(-(p * p.log()).sum() / log_n)

def overlay(gray, sal, alpha=0.55):
    rgb = np.repeat(gray[..., None], 3, axis=2)
    return np.clip((1 - alpha) * rgb + alpha * matplotlib.colormaps["plasma"](sal)[..., :3], 0, 1)

def record_attention(run, level, n_steps, seed=0):
    model = load_run(run); ext = model.policy.features_extractor; blk = ext.block
    if not hasattr(blk, "store_attn"):
        return None
    model.policy.set_training_mode(False); blk.store_attn = True; set_random_seed(seed)
    env = make_vec_env(level, 1, monitor=False); obs = env.reset()
    frames, sals, rows = [], [], []
    try:
        for t in range(n_steps):
            action, _ = model.predict(obs)
            frames.append(obs[0, :, :, -1].copy())
            sals.append(attention_saliency(blk.last_attn, ext.feat_chw[1:], obs.shape[1:3]))
            row_h, recv_h = entropies(blk.last_attn)
            obs, _, dones, infos = env.step(action)
            rows.append({"step": t, "x_pos": infos[0].get("x_pos"), "row_entropy": row_h, "received_entropy": recv_h})
    finally:
        blk.store_attn = False; env.close()
    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(DIRS["attention"] / f"attention_stats_{run}_{level}.csv", index=False)
    return np.array(frames), np.array(sals), stats_df, blk.gamma.item()

ATT_LEVEL = TEST_LEVELS[1] if len(TEST_LEVELS) > 1 else TEST_LEVELS[0]
att_rows, grids = [], []
for v in [v for v in VARIANTS if VARIANT_SPEC[v][0] in ("spatial", "sparse")]:
    run = run_name(v, SEEDS[0])
    if resolve_checkpoint(run) is None:
        continue
    gif = DIRS["gifs"] / f"attention_{run}_{ATT_LEVEL}.gif"
    res = record_attention(run, ATT_LEVEL, GIF_STEPS)
    if res is None:
        continue
    frames, sals, sdf, gamma = res
    att_rows.append({"Variant": LABELS[v], "Level": ATT_LEVEL, "Mean row entropy": sdf.row_entropy.mean(),
                     "Received-attention entropy": sdf.received_entropy.mean(), "γ": gamma})
    grids.append((v, frames, sals))
    if REEVALUATE or not gif.exists():
        imgs = [Image.fromarray((overlay(f / 255.0, s) * 255).astype(np.uint8)).resize((256, 240), Image.NEAREST)
                for f, s in zip(frames, sals)]
        imgs[0].save(gif, save_all=True, append_images=imgs[1:], duration=100, loop=0, optimize=True)
        print(f"saved → {RESULTS.name}/gifs/{gif.name}")
if att_rows:
    save_table(pd.DataFrame(att_rows), "attention_entropy",
               f"Attention focus on held-out level {ATT_LEVEL} (normalised entropy, 1 = uniform), seed {SEEDS[0]}", ",.3f")
    k = 6
    fig, axes = plt.subplots(len(grids), k, figsize=(2.6 * k, 2.6 * len(grids)), squeeze=False)
    for r, (v, frames, sals) in enumerate(grids):
        for c, i in enumerate(np.linspace(0, len(frames) - 1, k).astype(int)):
            axes[r, c].imshow(overlay(frames[i] / 255.0, sals[i])); axes[r, c].axis("off")
        axes[r, 0].text(-0.04, 0.5, LABELS[v], transform=axes[r, 0].transAxes, rotation=90, ha="right", va="center", fontsize=9)
    fig.suptitle(f"Attention received per position on held-out level {ATT_LEVEL} (each map rescaled)")
    fig.tight_layout(); save_fig(fig, "attention_comparison"); plt.show()

## 11. Inference latency

In [ ]:
NES_FPS = 60.0988; BUDGET_MS = 1000 * CONFIG["frame_skip"] / NES_FPS

@torch.no_grad()
def benchmark_latency(policy, device, n_iters=300, warmup=30):
    policy = policy.to(device).eval()
    obs = torch.randint(0, 256, (1, *OBS_SPACE.shape), dtype=torch.uint8, device=device)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    for _ in range(warmup):
        policy._predict(obs, deterministic=True)
    sync(); ts = []
    for _ in range(n_iters):
        t = time.perf_counter(); policy._predict(obs, deterministic=True); sync(); ts.append(1000 * (time.perf_counter() - t))
    return np.array(ts)

LAT_FILE = DIRS["tables"] / "inference_latency.csv"
if LAT_FILE.exists() and not REEVALUATE:
    lat = pd.read_csv(LAT_FILE); print("using saved latency table")
else:
    rows = []
    for dev in [torch.device("cpu")] + ([torch.device("cuda")] if torch.cuda.is_available() else []):
        for v in dict.fromkeys(VARIANTS):
            ms = benchmark_latency(build_policy(v), dev)
            rows.append({"Variant": LABELS[v], "Device": dev.type.upper(), "Mean (ms)": ms.mean(),
                         "p95 (ms)": np.percentile(ms, 95), "Real-time": bool(np.percentile(ms, 95) < BUDGET_MS)})
    lat = pd.DataFrame(rows)
save_table(lat, "inference_latency", f"Policy latency at batch 1 (budget {BUDGET_MS:.1f} ms)", ",.3f")

## 12. Results summary

In [ ]:
order = ["architecture_params", "random_reference", "generalization_summary", "generalization_statistics",
         "attention_entropy", "inference_latency"]
lines = [f"# Results: {TITLE}", "", f"Generated {time.strftime('%Y-%m-%d %H:%M')} · variants {VARIANTS} · seeds {SEEDS} · "
         f"{TOTAL_TIMESTEPS:,} steps per run · train levels {TRAIN_LEVELS} · held-out levels {TEST_LEVELS}", ""]
for name in order:
    p = DIRS["tables"] / f"{name}.md"
    if p.exists():
        lines += [p.read_text(), ""]
lines += ["## Figures", ""] + [f"![{p.stem}](figures/{p.name})" for p in sorted(DIRS["figures"].glob("*.png"))]
lines += ["", "## Animations", ""] + [f"![{p.stem}](gifs/{p.name})" for p in sorted(DIRS["gifs"].glob("*.gif"))]
(RESULTS / "summary.md").write_text("\n".join(lines) + "\n")
print(f"saved → {RESULTS.name}/summary.md")